# Stage 1 (MacroExpander) — Single FAVAR Flow

**구조**
- IN cond past   (3ch × past 52w)  : `tbill_wr, tbill_26w_lag, excess_liq_26w_lag`
- IN cond future (3ch × future 52w): `tbill_wr, tbill_26w_lag` (사용자 시나리오), `excess_liq_26w_lag` 는 **0 마스킹** (= train 평균)
- IN past target (1ch × past 52w)  : `excess_liq_yoy`
- OUT future target (1ch × future 52w): `excess_liq_yoy`
- Architecture: `MultiStepFAVARFlow K=2, d_model=64, n_heads=4, n_layers=2` (NLL only)
- z-score fix: test cond uses train mu/sd (calibration 정합)

**준비**: T4 또는 A100 런타임 → 셀 순차 실행.

## 1. Drive mount + repo clone/pull + cd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Colab Notebooks', exist_ok=True)
os.chdir('/content/drive/MyDrive/Colab Notebooks')

if not os.path.exists('homeostatic-market'):
    !git clone https://github.com/hwayobi2020/homeostatic-market.git
    print('Cloned fresh.')
else:
    os.chdir('/content/drive/MyDrive/Colab Notebooks/homeostatic-market')
    !git pull
    print('Pulled latest.')

os.chdir('/content/drive/MyDrive/Colab Notebooks/homeostatic-market/colab/dual_3ch')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

## 2. GPU 확인

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
    print('VRAM total     :', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 3. Stage 1 학습 (5 시드)

T4 ~50분, A100 ~25분.

In [ ]:
!mkdir -p result
!python train_stage1.py --seeds 42 123 777 0 99 2>&1 | tee result/stage1_multi_run.log

## 4. 결과 — 학습 곡선 + 시드별 summary

In [ ]:
import json, glob
import pandas as pd
import matplotlib.pyplot as plt

summaries = []
for f in sorted(glob.glob('result/stage1_seed*_summary.json')):
    with open(f) as fp:
        summaries.append(json.load(fp))
df_sum = pd.DataFrame(summaries)[['seed', 'best_epoch', 'val', 'test']]
print(df_sum.to_string(index=False))
print(f"\nval:  mean={df_sum.val.mean():+.4f} ± {df_sum.val.std():.4f}  median={df_sum.val.median():+.4f}")
print(f"test: mean={df_sum.test.mean():+.4f} ± {df_sum.test.std():.4f}  median={df_sum.test.median():+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for f in sorted(glob.glob('result/stage1_seed*_trainlog.csv')):
    seed = f.split('seed')[1].split('_')[0]
    log = pd.read_csv(f)
    axes[0].plot(log['epoch'], log['train'], label=f'seed{seed}', alpha=0.7)
    axes[1].plot(log['epoch'], log['val'],   label=f'seed{seed}', alpha=0.7)
    axes[2].plot(log['epoch'], log['test'],  label=f'seed{seed}', alpha=0.7)
axes[0].set_title('train NLL'); axes[0].grid(True); axes[0].legend(fontsize=8)
axes[1].set_title('val NLL');   axes[1].grid(True); axes[1].legend(fontsize=8)
axes[2].set_title('test NLL');  axes[2].grid(True); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Stage 2 학습 (paired vs Base K2_104)

**전제**: Stage 1 5 시드 ckpt (`stage1_seed{42,123,777,0,99}_best.pt`) 가 `result/` 에 있어야 함.

각 시드 S 의 Stage 2 는 **같은 시드 S 의 Stage 1 ckpt** 를 사용해 future `excess_liq_26w_lag` 자리에 deterministic mean prediction (z_future=0 inverse) → 26w rolling mean (Bridge) 로 채움. 그 외 architecture / cond / target / training 은 Base K2_104 와 동일.

T4 ~50분, A100 ~25분.

In [ ]:
!python train_stage2.py --seeds 42 123 777 0 99 2>&1 | tee result/stage2_multi_run.log